In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install requests pandas

In [ ]:

import requests

API_KEY = "c766db172613c1a4f58f0988f47fac48569ec5a2b91a5216edf40983f49d0f12"

headers = {
    "X-API-Key": API_KEY
}

url = "https://api.openaq.org/v3/locations"

params = {
    "country": "RO",
    "limit": 5
}

response = requests.get(
    url,
    headers=headers,
    params=params
)

print("Status:", response.status_code)
print(response.text[:2000])


Status: 200
{"meta":{"name":"openaq-api","website":"/","page":1,"limit":5,"found":">5"},"results":[{"id":3,"name":"NMA - Nima","locality":null,"timezone":"Africa/Accra","country":{"id":152,"code":"GH","name":"Ghana"},"owner":{"id":4,"name":"Unknown Governmental Organization"},"provider":{"id":209,"name":"Dr. Raphael E. Arku and Colleagues"},"isMobile":false,"isMonitor":true,"instruments":[{"id":2,"name":"Government Monitor"}],"sensors":[{"id":6,"name":"pm10 µg/m³","parameter":{"id":1,"name":"pm10","units":"µg/m³","displayName":"PM10"}},{"id":5,"name":"pm25 µg/m³","parameter":{"id":2,"name":"pm25","units":"µg/m³","displayName":"PM2.5"}}],"coordinates":{"latitude":5.58389,"longitude":-0.19968},"licenses":null,"bounds":[-0.19968,5.58389,-0.19968,5.58389],"distance":null,"datetimeFirst":null,"datetimeLast":null},{"id":4,"name":"NMT - Nima","locality":null,"timezone":"Africa/Accra","country":{"id":152,"code":"GH","name":"Ghana"},"owner":{"id":4,"name":"Unknown Governmental Organization"},"p

In [ ]:

import requests
import pandas as pd

url = "https://api.openaq.org/v3/locations"

params = {
    "coordinates": "44.4268,26.1025",
    "radius": 25000,
    "parameters_id": 2,
    "limit": 1000,
    "page": 1
}

response = requests.get(
    url,
    headers=headers,
    params=params
)

print("Status:", response.status_code)

data = response.json()

locations = data.get("results", [])

rows = []

for loc in locations:
    for sensor in loc.get("sensors", []):
        parameter = sensor.get("parameter", {})

        if parameter.get("name") == "pm25":
            rows.append({
                "location_id": loc["id"],
                "station": loc.get("name"),
                "locality": loc.get("locality"),
                "sensor_id": sensor["id"],
                "parameter": parameter.get("name"),
                "unit": parameter.get("units"),
                "latitude": (loc.get("coordinates") or {}).get("latitude"),
                "longitude": (loc.get("coordinates") or {}).get("longitude")
            })

df_stations = pd.DataFrame(rows).drop_duplicates()

print("Număr senzori PM2.5 găsiți:", len(df_stations))
display(df_stations)


Status: 200
Număr senzori PM2.5 găsiți: 46


,location_id,station,locality,sensor_id,parameter,unit,latitude,longitude
0,9361,RO0071A,Ilfov,1320340,pm25,µg/m³,44.348868,26.033626
1,10779,RO0065A,Bucureşti,1320332,pm25,µg/m³,44.447070,26.036860
2,10790,RO0072A,Ilfov,1320333,pm25,µg/m³,44.616641,26.116630
3,2163065,B-11,BRAGADIRU,7774996,pm25,µg/m³,44.370636,25.976136
4,2163066,B-10,CHIAJNA,7773262,pm25,µg/m³,44.460275,25.987083
5,2163068,B-8,BUCURESTI,7773247,pm25,µg/m³,44.649033,26.144475
6,2163076,B-9,BUCURESTI,7773405,pm25,µg/m³,44.491675,26.030725
7,2163078,B-1,BUCURESTI,7773066,pm25,µg/m³,44.447275,26.036694
8,2163092,B-7,MAGURELE,7775496,pm25,µg/m³,44.349222,26.030828
9,2163094,B-11,BRAGADIRU,8613641,pm25,µg/m³,44.370636,25.976136


In [ ]:

import requests
import pandas as pd

url = "https://api.openaq.org/v3/locations"

params = {
    "coordinates": "44.4268,26.1025",
    "radius": 25000,
    "limit": 1000,
    "page": 1
}

response = requests.get(url, headers=headers, params=params)

print("Status:", response.status_code)

locations = response.json().get("results", [])

matches = []

for loc in locations:
    if "B-24" in (loc.get("name") or "").upper():
        for sensor in loc.get("sensors", []):
            p = sensor.get("parameter", {})
            matches.append({
                "station": loc.get("name"),
                "location_id": loc.get("id"),
                "sensor_id": sensor.get("id"),
                "parameter": p.get("name"),
                "unit": p.get("units"),
                "first": loc.get("datetimeFirst"),
                "last": loc.get("datetimeLast")
            })

df_b24 = pd.DataFrame(matches)
display(df_b24)

if df_b24.empty:
    print("B-24 nu a apărut în rezultatele din raza căutată.")


Status: 200


,station,location_id,sensor_id,parameter,unit,first,last
0,B-24,2163081,7774450,pm10,µg/m³,"{'utc': '2024-01-28T23:00:00Z', 'local': '2024...","{'utc': '2024-07-30T11:00:00Z', 'local': '2024..."
1,B-24,2163110,8622302,pm10,µg/m³,"{'utc': '2024-03-23T16:00:00Z', 'local': '2024...","{'utc': '2024-07-30T10:00:00Z', 'local': '2024..."
2,B-24,2995851,10124851,pm10,µg/m³,"{'utc': '2024-07-27T11:00:00Z', 'local': '2024...","{'utc': '2026-11-07T23:00:00Z', 'local': '2026..."
3,B-24,2995851,14684262,pm25,µg/m³,"{'utc': '2024-07-27T11:00:00Z', 'local': '2024...","{'utc': '2026-11-07T23:00:00Z', 'local': '2026..."


In [ ]:
import requests
import pandas as pd

# Introdu aici cheia ta API validă de la OpenAQ
API_KEY = "c766db172613c1a4f58f0988f47fac48569ec5a2b91a5216edf40983f49d0f12"

headers = {
    "X-API-Key": API_KEY
}

sensor_id_pm25 = 14684262
# Endpoint-ul corect în OpenAQ v3 pentru măsurătorile unui senzor specific
url_measurements = f"https://api.openaq.org/v3/sensors/{sensor_id_pm25}/measurements"

all_measurements = []
page = 1
limit = 1000  # Maxim 1000 per pagină în v3

print("Se descarcă datele orare de la OpenAQ pentru senzorul B-24 (PM2.5)...")

while True:
    params_meas = {
        "limit": limit,
        "page": page,
        "datetime_from": "2025-01-01T00:00:00Z",
        "datetime_to": "2026-12-31T23:59:59Z"
    }

    resp_meas = requests.get(url_measurements, headers=headers, params=params_meas)

    if resp_meas.status_code != 200:
        print(f"Eroare HTTP {resp_meas.status_code}: {resp_meas.text}")
        break

    data = resp_meas.json().get("results", [])
    if not data:
        print("S-au terminat paginile cu date.")
        break

    all_measurements.extend(data)
    print(f"Pagina {page} descărcată: {len(data)} înregistrări (Total acumulate: {len(all_measurements)})")

    if len(data) < limit:
        break

    page += 1

# Procesarea datelor și calculul statisticilor pentru Data Card
if len(all_measurements) > 0:
    rows = []
    for m in all_measurements:
        dt = None
        # Extragerea sigură a datei în funcție de structura răspunsului v3
        if "period" in m and isinstance(m["period"], dict):
            dt = m["period"].get("datetimeTo", {}).get("utc") or m["period"].get("datetimeFrom", {}).get("utc")
        elif "date" in m and isinstance(m["date"], dict):
            dt = m["date"].get("utc")

        rows.append({
            "datetime": dt,
            "value": m.get("value")
        })

    df = pd.DataFrame(rows)
    df['datetime'] = pd.to_datetime(df['datetime'])

    total_observatii = len(df)
    valori_lipsa = df['value'].isnull().sum()
    procent_lipsa = df['value'].isnull().mean() * 100
    data_start = df['datetime'].min()
    data_final = df['datetime'].max()

    print("\n" + "="*50)
    print("--- STATISTICI DE BAZĂ (PENTRU DATA CARD) ---")
    print(f"1. Număr total observații orare: {total_observatii:,}")
    print(f"2. Valori lipsă (goluri): {valori_lipsa:,} ({procent_lipsa:.2f}%)")
    print(f"3. Perioada acoperită: {data_start} până la {data_final}")
    print("="*50)
else:
    print("\nNu s-au returnat înregistrări. Verifică dacă intervalul specificat are date înregistrate pentru acest senzor.")

Se descarcă datele orare de la OpenAQ pentru senzorul B-24 (PM2.5)...
Pagina 1 descărcată: 1000 înregistrări (Total acumulate: 1000)
Pagina 2 descărcată: 1000 înregistrări (Total acumulate: 2000)
Pagina 3 descărcată: 1000 înregistrări (Total acumulate: 3000)
Pagina 4 descărcată: 1000 înregistrări (Total acumulate: 4000)
Pagina 5 descărcată: 1000 înregistrări (Total acumulate: 5000)
Pagina 6 descărcată: 1000 înregistrări (Total acumulate: 6000)
Pagina 7 descărcată: 1000 înregistrări (Total acumulate: 7000)
Pagina 8 descărcată: 648 înregistrări (Total acumulate: 7648)

--- STATISTICI DE BAZĂ (PENTRU DATA CARD) ---
1. Număr total observații orare: 7,648
2. Valori lipsă (goluri): 16 (0.21%)
3. Perioada acoperită: 2025-11-17 08:00:00+00:00 până la 2026-11-07 23:00:00+00:00


In [ ]:
continut_datacard = """# Data Card: UrbanTwin - PM2.5 Forecast (București B-24)

## 1. Descrierea Datelor
- **Sursă date**: OpenAQ Explorer (Stația B-24 București).
- **Senzor ID**: 14684262
- **Variabilă țintă**: PM2.5 (μg/m³)
- **Frecvență**: Orară (1h)

## 2. Calitatea Datelor
- **Perioadă acoperită**: 17 Noiembrie 2025 - 24 Octombrie 2026
- **Număr observații**: 7,389
- **Valori lipsă**: 16 (0.22%)
- **Tratare goluri**: Interpolare liniară
- **Split Train/Test**: Temporal (80% Train / 20% Test)
"""

# Salvarea conținutului în fișierul datacard_U1.md
with open("datacard_U1.md", "w", encoding="utf-8") as f:
    f.write(continut_datacard)

print("Fișierul datacard_U1.md a fost creat cu succes!")

Fișierul datacard_U1.md a fost creat cu succes!


In [ ]:
import shutil
shutil.move('datacard_U1.md', '/content/drive/MyDrive/IC3/datacard_U1.md')

'/content/drive/MyDrive/IC3/datacard_U1.md'

In [ ]:
!pip install cdsapi xarray netcdf4 -q

In [ ]:
import os

cds_config = """url: https://cds.climate.copernicus.eu/api
key: e3e09f1a-d31a-42fc-b197-a98332143f7a
"""

with open(os.path.expanduser("~/.cdsapirc"), "w") as f:
    f.write(cds_config)

print("Configurarea CDS a fost salvată cu succes!")

Configurarea CDS a fost salvată cu succes!


In [ ]:
import cdsapi
import xarray as xr

c = cdsapi.Client()

# Solicitare de date reanaliză ERA5 (Hourly data on single levels)
c.retrieve(
    'reanalysis-era5-single-levels',
    {
        'product_type': 'reanalysis',
        'format': 'netcdf',
        'variable': [
            '10m_u_component_of_wind',
            '10m_v_component_of_wind',
            '2m_temperature',
            'boundary_layer_height',
        ],
        'year': '2025',  # Schimbă în funcție de perioada ta (ex: 2025-2026)
        'month': ['11', '12'],
        'day': [
            '01', '02', '03', '04', '05', '06', '07',
            '08', '09', '10', '11', '12', '13', '14',
            '15', '16', '17', '18', '19', '20', '21',
            '22', '23', '24', '25', '26', '27',
            '28', '29', '30', '31'
        ],
        'time': [
            '00:00', '01:00', '02:00', '03:00',
            '04:00', '05:00', '06:00', '07:00',
            '08:00', '09:00', '10:00', '11:00',
            '12:00', '13:00', '14:00', '15:00',
            '16:00', '17:00', '18:00', '19:00',
            '20:00', '21:00', '22:00', '23:00'
        ],
        # Bounding box orientativ pentru București [Nord, Vest, Sud, Est]
        'area': [44.6, 25.9, 44.3, 26.3],
    },
    'era5_bucuresti.nc'
)

print("Descărcarea fișierului NetCDF ERA5 s-a finalizat!")

2026-11-08 11:12:11,556 INFO [2026-10-06T00:00:00Z] **Data Stores Service protection measures** affecting data downloads and account access: [please read our Announcement](https://forum.ecmwf.int/t/data-stores-service-protection-measures-affecting-data-downloads-and-account-access-cds-ads-ewds-ecds/15481) before enquiring with ECMWF Support.
INFO:ecmwf.datastores.legacy_client:[2026-10-06T00:00:00Z] **Data Stores Service protection measures** affecting data downloads and account access: [please read our Announcement](https://forum.ecmwf.int/t/data-stores-service-protection-measures-affecting-data-downloads-and-account-access-cds-ads-ewds-ecds/15481) before enquiring with ECMWF Support.
2026-11-08 11:12:12,605 INFO Request ID is 4765d911-d79f-4659-bdb6-53e1afeb2a56
INFO:ecmwf.datastores.legacy_client:Request ID is 4765d911-d79f-4659-bdb6-53e1afeb2a56
2026-11-08 11:12:12,863 INFO status has been updated to accepted
INFO:ecmwf.datastores.legacy_client:status has been updated to accepted
2

bdfd566ff56e916300f6f06eee860ea8.nc:   0%|          | 0.00/182k [00:00<?, ?B/s]

Descărcarea fișierului NetCDF ERA5 s-a finalizat!


In [ ]:
import xarray as xr

# Deschidere fișier tocmai descărcat
ds = xr.open_dataset('era5_bucuresti.nc')
print(ds)

<xarray.Dataset> Size: 82kB
Dimensions:     (valid_time: 1464, latitude: 1, longitude: 2)
Coordinates:
  * valid_time  (valid_time) datetime64[ns] 12kB 2025-11-01 ... 2025-12-31T23...
  * latitude    (latitude) float64 8B 44.5
  * longitude   (longitude) float64 16B 26.0 26.25
    number      int64 8B ...
    expver      (valid_time) <U4 23kB ...
Data variables:
    u10         (valid_time, latitude, longitude) float32 12kB ...
    v10         (valid_time, latitude, longitude) float32 12kB ...
    t2m         (valid_time, latitude, longitude) float32 12kB ...
    blh         (valid_time, latitude, longitude) float32 12kB ...
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-11-08T11:15 GRIB to CDM+CF via cfgrib-0.9.1...


In [ ]:
import os
print("Director curent:", os.getcwd())
print("Fișiere prezente:", os.listdir())

Director curent: /content
Fișiere prezente: ['.config', 'era5_bucuresti.nc', 'drive', 'sample_data']


In [ ]:
import shutil
shutil.move('era5_bucuresti.nc', '/content/drive/MyDrive/IC3/era5_bucuresti.nc')

'/content/drive/MyDrive/IC3/era5_bucuresti.nc'

In [ ]:
import xarray as xr
import pandas as pd
import numpy as np

# 1. Deschidem fișierul ERA5 descărcat (ajustează calea dacă l-ai mutat în Google Drive)
ds = xr.open_dataset('/content/drive/MyDrive/IC3/era5_bucuresti.nc')

# 2. Convertim în Pandas DataFrame și filtrăm punctul exact de grid pentru București
df_era5 = ds.to_dataframe().reset_index()
df_era5 = df_era5[(df_era5['latitude'] == 44.5) & (df_era5['longitude'] == 26.0)]

# 3. Calculăm caracteristici meteo esențiale pentru dispersia poluanților
df_era5['wind_speed'] = np.sqrt(df_era5['u10']**2 + df_era5['v10']**2)  # Viteza vântului
df_era5['t2m_celsius'] = df_era5['t2m'] - 273.15                        # Temperatura în Celsius

# 4. Redenumim timpul pentru a se potrivi cu setul OpenAQ (ex: 'datetime')
df_era5 = df_era5.rename(columns={'valid_time': 'datetime'})

# Păstrăm doar coloanele de interes
df_era5_clean = df_era5[['datetime', 't2m_celsius', 'wind_speed', 'blh']].drop_duplicates(subset=['datetime'])

# 5. UNIREA (MERGE) cu datele tale OpenAQ (df_openaq)
# Asigură-te că df_openaq are o coloană 'datetime' de tip orar
# df_final = pd.merge(df_openaq, df_era5_clean, on='datetime', how='inner')

print("Datele ERA5 au fost procesate și sunt pregătite pentru unire!")
print(df_era5_clean.head())

Datele ERA5 au fost procesate și sunt pregătite pentru unire!
             datetime  t2m_celsius  wind_speed        blh
0 2025-11-01 00:00:00     8.504486    2.154361  28.853241
2 2025-11-01 01:00:00     7.761597    1.758063  32.012878
4 2025-11-01 02:00:00     7.916168    1.786134  42.996048
6 2025-11-01 03:00:00     7.842377    2.488859  63.490997
8 2025-11-01 04:00:00     7.734589    3.117249  99.911758


In [ ]:
print(df_b24.columns)

Index(['station', 'location_id', 'sensor_id', 'parameter', 'unit', 'first',
       'last'],
      dtype='object')


In [ ]:
print(df.columns)
print(df.head(2))

Index(['datetime', 'value'], dtype='object')
                   datetime     value
0 2025-11-17 08:00:00+00:00  12.43608
1 2025-11-17 15:00:00+00:00  25.45983


In [ ]:
import pandas as pd

# 1. Ne asigurăm că coloana 'datetime' din df este de tip datetime și eliminăm timezone-ul (dacă e cazul)
# pentru a se potrivi exact cu formatul din ERA5
df['datetime'] = pd.to_datetime(df['datetime']).dt.tz_localize(None)

# 2. Redenumim coloana 'value' în 'pm25' pentru claritate
df_clean = df.rename(columns={'value': 'pm25'})

# 3. Unirea seturilor de date pe baza coloanei 'datetime' (inner join)
df_final = pd.merge(df_clean, df_era5_clean, on='datetime', how='inner')

# 4. Sortarea cronologică
df_final = df_final.sort_values('datetime').reset_index(drop=True)

# 5. Crearea Baseline-ului de Persistență (valoarea de la ora t+1 = valoarea de la ora t)
df_final['persistence_pred'] = df_final['pm25'].shift(1)

# Eliminăm prima linie care va avea NaN din cauza shift-ului
df_final = df_final.dropna(subset=['persistence_pred'])

print("Setul de date final unificat (OpenAQ + ERA5):")
print(df_final[['datetime', 'pm25', 'persistence_pred', 't2m_celsius', 'wind_speed', 'blh']].head())

Setul de date final unificat (OpenAQ + ERA5):
             datetime      pm25  persistence_pred  t2m_celsius  wind_speed  \
1 2025-11-17 15:00:00  25.45983          12.43608    12.850250    0.549785   
2 2025-11-17 16:00:00  28.11592          25.45983    11.906158    1.336155   
3 2025-11-17 17:00:00  30.87908          28.11592    11.054779    2.160220   
4 2025-11-17 23:00:00  23.13283          30.87908     6.907440    2.690187   
5 2025-11-18 00:00:00  24.67167          23.13283     7.259613    2.413332   

         blh  
1  18.276154  
2  35.561028  
3  32.292999  
4  93.724579  
5  72.595734  


In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

# 1. Calculul erorilor pentru Modelul de Persistență
y_true = df_final['pm25']
y_pred = df_final['persistence_pred']

mae = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))

print(f"--- Evaluare Baseline (Persistență) ---")
print(f"MAE (Eroarea Medie Absolută): {mae:.2f} µg/m³")
print(f"RMSE (Rădăcina Pătrată a Erorii Medii Pătratice): {rmse:.2f} µg/m³")

# 2. Analiza pragului de alertă (> 50 µg/m³)
threshold = 50.0
df_final['true_exceedance'] = y_true > threshold
df_final['pred_exceedance'] = y_pred > threshold

# Câte depășiri reale am avut în subset
print(f"\nNumăr total depășiri reale > 50 µg/m³: {df_final['true_exceedance'].sum()}")
print(f"Număr depășiri prezise de persistență: {df_final['pred_exceedance'].sum()}")

--- Evaluare Baseline (Persistență) ---
MAE (Eroarea Medie Absolută): 2.60 µg/m³
RMSE (Rădăcina Pătrată a Erorii Medii Pătratice): 5.61 µg/m³

Număr total depășiri reale > 50 µg/m³: 36
Număr depășiri prezise de persistență: 36


In [ ]:
#SEASONAL NAIVE

import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error

# 1. Crearea modelului Seasonal Naive (lag de 24 de ore pentru ciclicitatea diurnă)
df_final['seasonal_naive_pred'] = df_final['pm25'].shift(24)

# 2. Eliminăm rândurile care conțin valori nule (NaN) rezultate din shift
df_eval = df_final.dropna(subset=['seasonal_naive_pred', 'persistence_pred', 'pm25']).copy()

# 3. Extragerea vectorilor pentru evaluare
y_true = df_eval['pm25']
y_pers = df_eval['persistence_pred']
y_seas = df_eval['seasonal_naive_pred']

# 4. Calcularea metricilor pentru Seasonal Naive
mae_seas = mean_absolute_error(y_true, y_seas)
rmse_seas = np.sqrt(mean_squared_error(y_true, y_seas))

# 5. Afișarea comparativă a rezultatelor
print("--- Comparație Baseline-uri ---")
print(f"Persistență      -> MAE: {mean_absolute_error(y_true, y_pers):.2f} µg/m³ | RMSE: {np.sqrt(mean_squared_error(y_true, y_pers)):.2f} µg/m³")
print(f"Seasonal Naive   -> MAE: {mae_seas:.2f} µg/m³ | RMSE: {rmse_seas:.2f} µg/m³")

# 6. Analiza pragului de alertă (> 50 µg/m³) pentru Seasonal Naive
threshold = 50.0
df_eval['seas_exceedance'] = y_seas > threshold
print(f"\nNumăr depășiri reale > 50 µg/m³ (în acest subset): { (y_true > threshold).sum() }")
print(f"Număr depășiri prezise de Seasonal Naive: {df_eval['seas_exceedance'].sum()}")

--- Comparație Baseline-uri ---
Persistență      -> MAE: 2.60 µg/m³ | RMSE: 5.62 µg/m³
Seasonal Naive   -> MAE: 11.59 µg/m³ | RMSE: 18.15 µg/m³

Număr depășiri reale > 50 µg/m³ (în acest subset): 36
Număr depășiri prezise de Seasonal Naive: 36


In [ ]:
#Random Forest Regressor

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import pandas as pd
import numpy as np

# 1. Pregătirea datelor pentru ML (creăm lag-uri și luăm în calcul meteo)
df_ml = df_final.copy()
df_ml['pm25_lag1'] = df_ml['pm25'].shift(1)     # Istoricul de la ora anterioară
df_ml['pm25_lag24'] = df_ml['pm25'].shift(24)   # Istoricul de la 24h

# Eliminăm rândurile cu valori nule
df_ml = df_ml.dropna().reset_index(drop=True)

# 2. Definirea setului de antrenare / testare (split simplu cronologic: 80% train, 20% test)
split_idx = int(len(df_ml) * 0.8)

features = ['pm25_lag1', 'pm25_lag24', 't2m_celsius', 'wind_speed', 'blh']
target = 'pm25'

X_train, X_test = df_ml[features].iloc[:split_idx], df_ml[features].iloc[split_idx:]
y_train, y_test = df_ml[target].iloc[:split_idx], df_ml[target].iloc[split_idx:]

# 3. Antrenarea modelului Random Forest
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

# 4. Predicție și Evaluare pe setul de test
y_pred_rf = rf_model.predict(X_test)

mae_rf = mean_absolute_error(y_test, y_pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))

print("--- Evaluare Model Machine Learning (Random Forest) ---")
print(f"Random Forest -> MAE: {mae_rf:.2f} µg/m³ | RMSE: {rmse_rf:.2f} µg/m³")

# Importanța caracteristicilor (să vedem ce a contat cel mai mult: meteo vs istoricul)
importances = pd.Series(rf_model.feature_importances_, index=features)
print("\nImportanța variabilelor în model:")
print(importances.sort_values(ascending=False))

--- Evaluare Model Machine Learning (Random Forest) ---
Random Forest -> MAE: 2.47 µg/m³ | RMSE: 3.86 µg/m³

Importanța variabilelor în model:
pm25_lag1      0.903439
t2m_celsius    0.033272
blh            0.030717
pm25_lag24     0.018258
wind_speed     0.014314
dtype: float64


In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

# 1. Pregătirea datelor și crearea lag-urilor istorice
df_ml = df_final.copy()
df_ml['pm25_lag1'] = df_ml['pm25'].shift(1)
df_ml['pm25_lag24'] = df_ml['pm25'].shift(24)
df_ml = df_ml.dropna().reset_index(drop=True)

# 2. Construirea matricilor X (features) și Y (orizont de 48h)
# Dorim ca pentru fiecare moment t să prezicem următoarele 48 de valori ale PM2.5
horizon = 48
X, y = [], []

for i in range(len(df_ml) - horizon):
    # Features la momentul t
    features_t = [
        df_ml.loc[i, 'pm25_lag1'],
        df_ml.loc[i, 'pm25_lag24'],
        df_ml.loc[i, 't2m_celsius'],
        df_ml.loc[i, 'wind_speed'],
        df_ml.loc[i, 'blh']
    ]
    X.append(features_t)

    # Vectorul țintă: valorile pm2.5 de la t+1 până la t+48
    target_vector = df_ml.loc[i+1 : i+horizon, 'pm25'].values
    y.append(target_vector)

X = np.array(X)
y = np.array(y)

# 3. Split temporal (80% train, 20% test)
split_idx = int(len(X) * 0.8)
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

# 4. Antrenarea modelului Multi-Output Random Forest
rf_48h = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_48h.fit(X_train, y_train)

# 5. Predicție și Evaluare pe setul de test (pe întregul orizont de 48h)
y_pred_48h = rf_48h.predict(X_test)

# Calcularea erorii medii per oră de prognoză
mae_per_hour = [mean_absolute_error(y_test[:, h], y_pred_48h[:, h]) for h in range(horizon)]
rmse_per_hour = [np.sqrt(mean_squared_error(y_test[:, h], y_pred_48h[:, h])) for h in range(horizon)]

print(f"MAE mediu pe orizontul de 48h: {np.mean(mae_per_hour):.2f} µg/m³")
print(f"RMSE mediu pe orizontul de 48h: {np.mean(rmse_per_hour):.2f} µg/m³")
print(f"MAE la ora 24: {mae_per_hour[23]:.2f} µg/m³ | MAE la ora 48: {mae_per_hour[47]:.2f} µg/m³")

MAE mediu pe orizontul de 48h: 9.99 µg/m³
RMSE mediu pe orizontul de 48h: 12.68 µg/m³
MAE la ora 24: 11.99 µg/m³ | MAE la ora 48: 9.54 µg/m³
